In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql://tongli1997@db.pcic.uvic.ca:5433/crmp?keepalives=1&keepalives_idle=300&keepalives_interval=300&keepalives_count=9&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_attribution = df[df["ISSUE"].str.contains("Attribution", na=False)]

len(df_attribution)

df_attribution =  df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df_attribution["Network ID"] = pd.to_numeric(df_attribution["Network ID"], errors="coerce").astype("Int64")
df_attribution["Station ID"] = pd.to_numeric(df_attribution["Station ID"], errors="coerce").astype("Int64")
df_attribution


,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,Attribution,2322,12,BC-FWx-> BC-TS,945,TS NAKA CREEK,"115.2144 W, 49.6673 N, Elev. 1051 m -> 126.433..."
1,Attribution,2633,9,BC-Air -> MVRD-AIR,E246240 -> T034,Abbotsford Airport - Walmsley Road,"122.343 W, 49.0235 N, Elev. 65 m"
2,Attribution,12610,9,BC-Air -> MVRD-AIR,New Westminster Sapperton Park -> T046,NA -> New Westminister,"122.894487 W, 49.227045 N, Elev. null m -> 122..."
3,Attribution,12608,9,BC-Air -> MVRD-AIR,Surrey East -> T015,NA -> Surrey East,"122.6942 W, 49.1328 N, Elev. null m -> 122.695..."
4,Attribution,12529,9,BC-Air -> MVRD-AIR,Abbotsford A Columbia Street -> T045,NA -> Abbotsford Airport,"122.3266 W, 49.0215 N, Elev. null m -> 122.326..."
...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,4B15P,Lu Lake,"126.3 W, 54.2 N, Elev. 1310 m"
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,1A02P,McBride (Upper),"120.333 W, 53.3 N, Elev. 1620 m"
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,2F05P,Mission Creek,"118.95 W, 49.95 N, Elev. 1780 m"
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,2A21P,Molson Creek,"118.233 W, 52.2333 N, Elev. 1980 m"


In [5]:
df_attribution = df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name']]
df_attribution

def split_network_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
    df_attribution['Network Name'].apply(split_network_name)
)

df_attribution['new_net_name'] = (
    df_attribution['new_net_name']
    .replace(['MVRD-AIR', 'MVRD'], 'MVRD-AQ')
)


df_attribution 

/tmp/ipykernel_11969/811755646.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
/tmp/ipykernel_11969/811755646.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2


### select the netwok_id based on new_net_name, summarize back to the table 

In [6]:
# Create an empty column first
df_attribution['new_network_id'] = None

with engine.connect() as conn:
    for idx, row in df_attribution.iterrows():
        # Query network_id for this row's new_net_name
        res = conn.execute(
            sa.text("""
                SELECT network_id
                FROM meta_network
                WHERE network_display_name = :network_name
            """),
            {"network_name": row['new_net_name']}
        ).mappings().fetchone()

        if res is not None:
            df_attribution.at[idx, 'new_network_id'] = res['network_id']
        else:
            print(f"❌ Network '{row['new_net_name']}' not found for station {row['Station ID']}")

df_attribution

,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
...,...,...,...,...,...,...,...,...
131,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
132,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
133,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
134,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5


In [7]:
df_attribution['new_network'] = df_attribution['new_network_id']> 50
df_attri_new_net = df_attribution[df_attribution['new_network'] == True]
df_attri_old_net = df_attribution[df_attribution['new_network'] == False]
df_attri_old_net = df_attri_old_net.drop(columns=['n_net_names'])
df_attri_old_net = df_attri_old_net.rename(columns={'Network ID': 'old_network_id'})

df_attri_new_net = df_attri_new_net.reset_index(drop=True)
df_attri_old_net

,ISSUE,Station ID,old_network_id,Network Name,old_net_name,new_net_name,new_network_id,new_network
12,Attribution,2569,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
13,Attribution,2532,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
14,Attribution,2551,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
15,Attribution,2521,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
16,Attribution,2567,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
17,Attribution,2522,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
18,Attribution,2548,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
19,Attribution,2547,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
20,Attribution,2541,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False
21,Attribution,2565,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,5,False


In [8]:
from sqlalchemy import text
import pandas as pd
from sqlalchemy.engine import Engine
from typing import Optional, List

def compare_network_vars(
    engine: Engine,
    df_attri_old_net: pd.DataFrame,
    old_network_id: int,
    new_network_id: int,
    station_ids: Optional[List[int]] = None
) -> pd.DataFrame:
    """
    Compare variables used by stations in an old network to variables in a new network.

    Parameters
    ----------
    engine : Engine
        SQLAlchemy engine connected to the database.
    df_attri_old_net : pd.DataFrame
        DataFrame containing at least 'Station ID' and 'old_network_id'.
    old_network_id : int
        old_network_id of the old network whose stations we want to check.
    new_network_id : int
        old_network_id of the target network to compare variables against.
    station_ids : Optional[List[int]]
        List of station IDs to check. If None, all stations in the old network are used.

    Returns
    -------
    pd.DataFrame
        DataFrame showing old station variables, matching new network vars, 
        and a boolean column 'exists_in_new_net'.
    """
    # Get station IDs if not provided
    if station_ids is None:
        station_ids = df_attri_old_net.loc[
            (df_attri_old_net['old_network_id'] == old_network_id) &
            (df_attri_old_net['new_network_id'] == new_network_id),
            'Station ID'
        ].dropna().tolist()

    if not station_ids:
        print("⚠️ No stations found for the old network.")
        return pd.DataFrame()

    print(station_ids)

    # --- Load variables used by the old stations ---
    SQL_STATION_VARS = text("""
    SELECT DISTINCT
           v.vars_id,
           v.unit,
           v.precision,
           v.standard_name,
           v.cell_method,
           v.long_description,
           v.net_var_name,
           v.display_name,
           v.short_name
    FROM obs_raw o
    JOIN meta_history h ON o.history_id = h.history_id
    JOIN meta_station s ON h.station_id = s.station_id
    JOIN meta_vars v ON o.vars_id = v.vars_id
    WHERE s.station_id = ANY(:station_ids)
    ORDER BY v.vars_id
    """)

    with engine.connect() as conn:
        df_vars_old_stations = pd.read_sql(SQL_STATION_VARS, conn, params={"station_ids": station_ids})

    if df_vars_old_stations.empty:
        print("⚠️ No variables found for the old stations.")
        return pd.DataFrame()

    # --- Load variables in the new network ---
    SQL_VARS_NEW = text("""
    SELECT DISTINCT
           v.vars_id,
           v.unit,
           v.precision,
           v.standard_name,
           v.cell_method,
           v.long_description,
           v.net_var_name,
           v.display_name,
           v.short_name
    FROM meta_vars v
    JOIN meta_network n ON v.network_id = n.network_id
    WHERE n.network_id = :network_id
    ORDER BY v.vars_id

    """)

    new_net_vars = pd.read_sql(SQL_VARS_NEW, engine, params={"network_id": new_network_id})

    # --- Compare variables ---
    VAR_MATCH_COLS = [
        "unit",
        "precision",
        "standard_name",
        "cell_method",
        "long_description",
        "net_var_name",
        "display_name",
        "short_name",
    ]

    df_compare = df_vars_old_stations.merge(
        new_net_vars[VAR_MATCH_COLS + ["vars_id"]],
        on=VAR_MATCH_COLS,
        how="left",
        suffixes=("", "_new_net"),
        indicator=True
    )

    # Mark existence in new network
    df_compare["exists_in_new_net"] = df_compare["_merge"] == "both"
    df_compare["vars_id_new_net"] = df_compare["vars_id_new_net"]

    return df_compare, df_vars_old_stations, new_net_vars

In [9]:
from pycds import Variable
from sqlalchemy.orm import Session
from typing import Tuple, Dict, List

def create_vars_for_network(
    df_vars: pd.DataFrame,
    network_id: int,
    session: Session
) -> Tuple[List[int], Dict[int, int]]:
    """
    Create variables in meta_vars for a given network based on a DataFrame,
    and return the list of new variable IDs and a mapping from old_vars_id -> new_vars_id.
    
    Parameters
    ----------
    df_vars : pd.DataFrame
        DataFrame containing the variable info with columns:
        ['vars_id', 'unit', 'precision', 'standard_name', 'cell_method',
         'long_description', 'net_var_name', 'display_name', 'short_name']
    network_id : int
        The network_id to which these variables will belong.
    session : sqlalchemy.orm.Session
        The active SQLAlchemy session.
    
    Returns
    -------
    vars_created : List[int]
        List of newly created variable IDs.
    vars_map : Dict[int, int]
        Mapping from old_vars_id to new_vars_id.
    """
    vars_created = []
    vars_map = {}

    for _, row in df_vars.iterrows():
        v = Variable(
            network_id=network_id,
            unit=row["unit"],
            precision=row["precision"],
            standard_name=row["standard_name"],
            cell_method=row["cell_method"],
            description=row["long_description"],
            name=row["net_var_name"],
            display_name=row["display_name"],
            short_name=row["short_name"],
        )

        session.add(v)
        session.flush()  # assigns vars_id immediately

        vars_created.append(v.id)
        vars_map[row["vars_id"]] = v.id

    session.commit()
    print(f"✅ Created {len(vars_created)} variables for network {network_id}")

    return vars_created, vars_map


In [10]:
import json
from sqlalchemy import text
from sqlalchemy.engine import Engine
from typing import Dict


SQL_UPDATE_VARS_BULK = text("""
UPDATE obs_raw o
SET vars_id = m.new_vars_id
FROM meta_history h
JOIN meta_station s
  ON h.station_id = s.station_id
CROSS JOIN json_to_recordset(:pairs_json)
     AS m(old_vars_id INT, new_vars_id INT)
WHERE o.history_id = h.history_id
  AND s.station_id = :station_id
  AND o.vars_id = m.old_vars_id
""")


def remap_obs_vars_for_station(
    engine: Engine,
    station_id: int,
    vars_map: Dict[int, int],
) -> int:
    """
    Bulk-remap obs_raw.vars_id for a given station using an old→new vars_id map.

    Parameters
    ----------
    engine : sqlalchemy.engine.Engine
        SQLAlchemy engine.
    station_id : int
        Station ID whose observations will be updated.
    vars_map : dict[int, int]
        Mapping of old_vars_id → new_vars_id.

    Returns
    -------
    int
        Number of rows updated.
    """
    if not vars_map:
        print("⚠️ vars_map is empty — nothing to update.")
        return 0


    # Only include old_vars_id that differ from current obs_raw
    rows_to_update = [
        (old, new) for old, new in vars_map.items()
        if old != new
    ]

    if not rows_to_update:
        print("🔹 All vars already mapped — nothing to update")
        return 0

    rows_to_update_json = json.dumps(
        [{"old_vars_id": old, "new_vars_id": new} for old, new in rows_to_update]
    )


    # Ensure plain Python int (avoid numpy.int64 issues)
    station_id = int(station_id)

    with engine.begin() as conn:
        res = conn.execute(
            SQL_UPDATE_VARS_BULK,
            {
                "pairs_json": rows_to_update_json,
                "station_id": station_id,
            }
        )

    print(
        f"✅ vars_id remap completed | "
        f"station={station_id} | "
        f"mappings={len(vars_map)} | "
        f"rows_updated={res.rowcount}"
    )

    return res.rowcount

In [11]:
def migrate_network_vars_and_remap_obs(
    engine,
    session,
    df_attri_old_net,
    old_network_id: int,
    new_network_id: int,
    dry_run: bool = False,
):
    """
    1. Compare vars between old and new network
    2. Create missing vars in new network (if needed)
    3. Build vars_id mapping
    4. Remap obs_raw vars_id station by station

    Parameters
    ----------
    dry_run : bool
        If True, do NOT update database — only print what would happen.
    """

    print(f"\n=== Migrating vars {old_network_id} → {new_network_id} ===")

    # -------------------------------------------------
    # Step 1: Compare network vars
    # -------------------------------------------------
    df_compare, _, _ = compare_network_vars(
        engine=engine,
        df_attri_old_net=df_attri_old_net,
        old_network_id=old_network_id,
        new_network_id=new_network_id,
    )

    df_compare["vars_id_new_net"] = (
        pd.to_numeric(df_compare["vars_id_new_net"], errors="coerce")
        .astype("Int64")
    )

    # -------------------------------------------------
    # Step 2: Create missing vars if needed
    # -------------------------------------------------
    df_missing = df_compare[df_compare["_merge"] != "both"].copy()

    vars_map_created = {}

    if not df_missing.empty:
        print(f"Creating {len(df_missing)} missing vars in new network...")

        if not dry_run:
            vars_created, vars_map_created = create_vars_for_network(
                df_missing,
                network_id=new_network_id,
                session=session,
            )
            print("Created vars:", vars_map_created)
        else:
            print("Dry run: vars would be created here.")

    # -------------------------------------------------
    # Step 3: Fill new vars_id safely
    # -------------------------------------------------
    if vars_map_created:
        df_compare["vars_id_new_net"] = (
            df_compare["vars_id_new_net"]
            .fillna(df_compare["vars_id"].map(vars_map_created))
        )

    # Safety check
    if df_compare["vars_id_new_net"].isna().any():
        raise ValueError("Some vars_id_new_net are still NaN. Aborting.")

    # Final mapping
    vars_map = {
        int(r["vars_id"]): int(r["vars_id_new_net"])
        for _, r in df_compare.iterrows()
    }

    print(f"Total vars mapped: {len(vars_map)}")

    # -------------------------------------------------
    # Step 4: Get stations
    # -------------------------------------------------
    subset = (
        df_attri_old_net.loc[
            (df_attri_old_net["old_network_id"] == old_network_id)
            & (df_attri_old_net["new_network_id"] == new_network_id)
        ]
        .reset_index(drop=True)
    )

    station_ids = subset["Station ID"].dropna().unique()

    print(f"Total stations to update: {len(station_ids)}")

    # -------------------------------------------------
    # Step 5: Remap obs station by station
    # -------------------------------------------------
    total_rows_updated = 0

    for station_id in station_ids:
        if dry_run:
            print(f"[Dry Run] Would update station {station_id}")
            continue

        rows_updated = remap_obs_vars_for_station(
            engine=engine,
            station_id=station_id,
            vars_map=vars_map,
        )

        print(f"Station {station_id}: {rows_updated} rows updated")
        total_rows_updated += rows_updated

    print(f"\nTotal rows updated: {total_rows_updated}")

    return {
        "vars_map": vars_map,
        "stations_updated": len(station_ids),
        "rows_updated": total_rows_updated,
    }

In [12]:
pair_1 = migrate_network_vars_and_remap_obs(
    engine=engine,
    session=session,
    df_attri_old_net=df_attri_old_net,
    old_network_id=14,
    new_network_id=5,
    dry_run=False,  # change to True for safety check
)

print(pair_1)


=== Migrating vars 14 → 5 ===
[2569, 2532, 2551, 2521, 2567, 2522, 2548, 2547, 2541, 2565, 2526, 2556, 2576, 2523, 2528, 2527, 2555, 2564, 2574, 2540, 2531, 2538, 2575, 2517, 2544, 12498, 2542, 2559, 2550, 2578, 2577, 2570, 2553, 2530, 12499, 12500, 2515, 2563, 2580, 12497, 2536, 2571, 2513, 2554, 2543, 2534]
Creating 8 missing vars in new network...
✅ Created 8 variables for network 5
Created vars: {660: 1327, 701: 1328, 702: 1329, 703: 1330, 704: 1331, 705: 1332, 706: 1333, 710: 1334}
Total vars mapped: 17
Total stations to update: 46
✅ vars_id remap completed | station=2569 | mappings=17 | rows_updated=481225
Station 2569: 481225 rows updated
✅ vars_id remap completed | station=2532 | mappings=17 | rows_updated=88845
Station 2532: 88845 rows updated
✅ vars_id remap completed | station=2551 | mappings=17 | rows_updated=441941
Station 2551: 441941 rows updated
✅ vars_id remap completed | station=2521 | mappings=17 | rows_updated=449419
Station 2521: 449419 rows updated
✅ vars_id rema

In [13]:
pair_2 = migrate_network_vars_and_remap_obs(
    engine=engine,
    session=session,
    df_attri_old_net=df_attri_old_net,
    old_network_id=12,
    new_network_id=15,
    dry_run=False,  # change to True for safety check
)

print(pair_2)


=== Migrating vars 12 → 15 ===
[13011, 13012, 13013]
Creating 5 missing vars in new network...
✅ Created 5 variables for network 15
Created vars: {496: 1344, 497: 1345, 498: 1346, 499: 1347, 500: 1348}
Total vars mapped: 5
Total stations to update: 3
✅ vars_id remap completed | station=13011 | mappings=5 | rows_updated=126299
Station 13011: 126299 rows updated
✅ vars_id remap completed | station=13012 | mappings=5 | rows_updated=136803
Station 13012: 136803 rows updated
✅ vars_id remap completed | station=13013 | mappings=5 | rows_updated=126038
Station 13013: 126038 rows updated

Total rows updated: 389140
{'vars_map': {496: 1344, 497: 1345, 498: 1346, 499: 1347, 500: 1348}, 'stations_updated': 3, 'rows_updated': 389140}


### In the new run, need to double check the manually matched variable id

In [14]:
old_network_id=12
new_network_id=11

df_compare, df_vars_old_stations, new_net_vars = compare_network_vars(
    engine=engine,
    df_attri_old_net=df_attri_old_net,
    old_network_id=old_network_id,
    new_network_id=new_network_id,
)


[11017, 12464]


In [15]:
df_compare

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,vars_id_new_net,_merge,exists_in_new_net
0,496,mm,None,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,precipitation,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,NaN,left_only,False
1,497,celsius,None,air_temperature,time: point,None,temperature,Temperature (Point),air_temperature_point,NaN,left_only,False
2,498,%,None,relative_humidity,time: mean,None,relative_humidity,Relative Humidity (Mean),relative_humidity_mean,NaN,left_only,False
3,499,km/h,None,wind_speed,time: mean,None,wind_speed,Wind Speed (Mean),wind_speed_mean,NaN,left_only,False
4,500,degree,None,wind_from_direction,time: mean,None,wind_direction,Wind Direction (Mean),wind_from_direction_mean,NaN,left_only,False
5,711,celsius,None,air_temperature,time: point,None,air_temp,Temperature,T,NaN,left_only,False
6,712,mm,None,thickness_of_rainfall_amount,time: sum (interval: 1 hour),None,rnfl_amt_pst1hr,Rainfall 1hr,1hr rain,NaN,left_only,False
7,713,%,None,relative_humidity,time: point,None,rel_hum,Relative Humidity,RH,NaN,left_only,False
8,714,km/h,None,wind_speed,time: mean,None,avg_wnd_spd_10m_pst10mts,Wind Speed,Wind Speed,NaN,left_only,False
9,715,degrees,None,wind_from_direction,time: mean,None,avg_wnd_dir_10m_pst10mts,Wind Direction,wdir,NaN,left_only,False


In [16]:
new_net_vars

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name
0,501,mm,None,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,Rainmm,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum
1,502,millibar,None,air_pressure,time: point,None,Pressurembar,Air Pressure (Point),air_pressure_point
2,503,celsius,None,air_temperature,time: point,None,TempC,Temperature (Point),air_temperature_point
3,504,%,None,relative_humidity,time: mean,None,RH,Relative Humidity (Mean),relative_humidity_mean
4,505,celsius,None,dew_point_temperature,time: mean,None,DewPtC,Dew Point Temperature (Mean),dew_point_temperature_mean
5,506,m s-1,None,wind_speed,time: mean,None,WindSpeedms,Wind Speed (Mean),wind_speed_mean
6,507,m s-1,None,wind_speed_of_gust,time: maximum,None,GustSpeedms,Wind Gust (Max.),wind_speed_of_gust_maximum
7,508,degree,None,wind_from_direction,time: mean,None,WindDirection,Wind Direction (Mean),wind_from_direction_mean
8,509,W m-2,None,surface_downwelling_shortwave_flux,time: mean,None,SolarRadiationWm,Incoming Shortwave,surface_downwelling_shortwave_flux_mean
9,510,%,None,volume_fraction_of_condensed_water_in_soil,time: point,None,Wetness,Soil Moisture,volume_fraction_of_condensed_water_in_soil_point


In [17]:

### Manually mapping
manual_map = {
    496: 501,
    497: 503,
    498: 504,
    499: 506,
    500: 508,
    719: 505,
    736: 832,
    # 711: 503,
    # 712: 501,
    # 713: 504,
    # 714: 506,
    # 715: 508,
}

mask = df_compare['vars_id'].isin(manual_map.keys())

df_compare.loc[mask, 'vars_id_new_net'] = (
    df_compare.loc[mask, 'vars_id'].map(manual_map)
)

df_compare['exists_in_new_net'] = df_compare['exists_in_new_net'].map({
    True: 'auto',
    False: 'missing'
}).astype('object')

df_compare.loc[mask, 'exists_in_new_net'] = 'manually'

df_compare["vars_id_new_net"] = (
    pd.to_numeric(df_compare["vars_id_new_net"], errors="coerce")
    .astype("Int64")
)
df_compare

df_matched = df_compare[
    df_compare["vars_id_new_net"].notna()
    & (df_compare["vars_id"] != df_compare["vars_id_new_net"])
]

df_matched

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,vars_id_new_net,_merge,exists_in_new_net
0,496,mm,None,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,precipitation,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,501,left_only,manually
1,497,celsius,None,air_temperature,time: point,None,temperature,Temperature (Point),air_temperature_point,503,left_only,manually
2,498,%,None,relative_humidity,time: mean,None,relative_humidity,Relative Humidity (Mean),relative_humidity_mean,504,left_only,manually
3,499,km/h,None,wind_speed,time: mean,None,wind_speed,Wind Speed (Mean),wind_speed_mean,506,left_only,manually
4,500,degree,None,wind_from_direction,time: mean,None,wind_direction,Wind Direction (Mean),wind_from_direction_mean,508,left_only,manually
11,719,celsius,None,dew_point_temperature,time: point,None,dwpt_temp,Dew Point Temperature,Td,505,left_only,manually
12,736,cm,None,surface_snow_thickness,time: point,None,snw_dpth,Snow Depth,sd,832,left_only,manually


In [18]:

vars_map = {
    int(r["vars_id"]): int(r["vars_id_new_net"])
    for _, r in df_matched.iterrows()
}

print(f"Total vars mapped: {len(vars_map)}")

# -------------------------------------------------
# Step 4: Get stations
# -------------------------------------------------
subset = (
    df_attri_old_net.loc[
        (df_attri_old_net["old_network_id"] == old_network_id)
        & (df_attri_old_net["new_network_id"] == new_network_id)
    ]
    .reset_index(drop=True)
)

station_ids = subset["Station ID"].dropna().unique()

# station_ids = [12464]



print(f"Total stations to update: {len(station_ids)}")

# -------------------------------------------------
# Step 5: Remap obs station by station
# -------------------------------------------------
total_rows_updated = 0

for station_id in station_ids:
    if dry_run:
        print(f"[Dry Run] Would update station {station_id}")
        continue

    rows_updated = remap_obs_vars_for_station(
        engine=engine,
        station_id=station_id,
        vars_map=vars_map,
    )

    print(f"Station {station_id}: {rows_updated} rows updated")
    total_rows_updated += rows_updated

print(f"\nTotal rows updated: {total_rows_updated}")

Total vars mapped: 7
Total stations to update: 2


NameError: name 'dry_run' is not defined

In [19]:
df_unmatched = df_compare[df_compare["vars_id_new_net"].isna()]
df_unmatched

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,vars_id_new_net,_merge,exists_in_new_net
5,711,celsius,None,air_temperature,time: point,None,air_temp,Temperature,T,<NA>,left_only,missing
6,712,mm,None,thickness_of_rainfall_amount,time: sum (interval: 1 hour),None,rnfl_amt_pst1hr,Rainfall 1hr,1hr rain,<NA>,left_only,missing
7,713,%,None,relative_humidity,time: point,None,rel_hum,Relative Humidity,RH,<NA>,left_only,missing
8,714,km/h,None,wind_speed,time: mean,None,avg_wnd_spd_10m_pst10mts,Wind Speed,Wind Speed,<NA>,left_only,missing
9,715,degrees,None,wind_from_direction,time: mean,None,avg_wnd_dir_10m_pst10mts,Wind Direction,wdir,<NA>,left_only,missing
10,718,mm,None,thickness_of_rainfall_amount,time: sum (interval: 24 hour),None,rnfl_amt_pst24hrs,Rainfall 24hr,rain_24hr,<NA>,left_only,missing


In [20]:

# -------------------------------------------------
# Step 2: Create missing vars if needed
# -------------------------------------------------
df_missing = df_unmatched

vars_map_created = {}

dry_run = False
if not df_missing.empty:
    print(f"Creating {len(df_missing)} missing vars in new network...")

    if not dry_run:
        vars_created, vars_map_created = create_vars_for_network(
            df_missing,
            network_id=new_network_id,
            session=session,
        )
        print("Created vars:", vars_map_created)
    else:
        print("Dry run: vars would be created here.")


Creating 6 missing vars in new network...
✅ Created 6 variables for network 11
Created vars: {711: 1349, 712: 1350, 713: 1351, 714: 1352, 715: 1353, 718: 1354}


In [21]:

# -------------------------------------------------
# Step 3: Fill new vars_id safely
# -------------------------------------------------
if vars_map_created:
    df_unmatched["vars_id_new_net"] = (
        df_unmatched["vars_id_new_net"]
        .fillna(df_unmatched["vars_id"].map(vars_map_created))
    )

# Safety check
if df_unmatched["vars_id_new_net"].isna().any():
    raise ValueError("Some vars_id_new_net are still NaN. Aborting.")

# Final mapping
vars_map = {
    int(r["vars_id"]): int(r["vars_id_new_net"])
    for _, r in df_unmatched.iterrows()
}

print(f"Total vars mapped: {len(vars_map)}")

# -------------------------------------------------
# Step 4: Get stations
# -------------------------------------------------
subset = (
    df_attri_old_net.loc[
        (df_attri_old_net["old_network_id"] == old_network_id)
        & (df_attri_old_net["new_network_id"] == new_network_id)
    ]
    .reset_index(drop=True)
)

station_ids = subset["Station ID"].dropna().unique()

print(f"Total stations to update: {len(station_ids)}")


Total vars mapped: 6
Total stations to update: 2


/tmp/ipykernel_11969/3179923881.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unmatched["vars_id_new_net"] = (


In [22]:

# -------------------------------------------------
# Step 5: Remap obs station by station
# -------------------------------------------------
total_rows_updated = 0

for station_id in station_ids:
    if dry_run:
        print(f"[Dry Run] Would update station {station_id}")
        continue

    rows_updated = remap_obs_vars_for_station(
        engine=engine,
        station_id=station_id,
        vars_map=vars_map,
    )

    print(f"Station {station_id}: {rows_updated} rows updated")
    total_rows_updated += rows_updated

print(f"\nTotal rows updated: {total_rows_updated}")



✅ vars_id remap completed | station=11017 | mappings=6 | rows_updated=0
Station 11017: 0 rows updated
✅ vars_id remap completed | station=12464 | mappings=6 | rows_updated=236557
Station 12464: 236557 rows updated

Total rows updated: 236557
